In [ ]:
import tempfile
import os
import subprocess
import shutil
from glob import glob
import json
import os
from openai import OpenAI


from pydantic import BaseModel, Field

In [34]:
os.environ['NEBIUS_API_KEY'] = "v1.CmQKHHN0YXRpY2tleS1lMDBubmp3bm52ZWdmOW1xcjESIXNlcnZpY2VhY2NvdW50LWUwMHhycnl0eWU5NTlhMDZqZTIMCIm28swGENCekq0COgwIiLmKmAcQgI-OxwJAAloDZTAw.AAAAAAAAAAEDvlmf4zvsU99xk2-GEj7IdhuDXTD_ImLWbB4yDuBWznxcvkyhoxr6rhO9bbOOk0Wbpjh_Ire5q-j68lv22B8P"

In [ ]:
client = OpenAI(
    base_url="https://api.tokenfactory.nebius.com/v1/",
    api_key=os.environ.get("NEBIUS_API_KEY")
)


In [13]:
github_url = 'https://github.com/handong1587/handong1587.github.io'
clone_path = '/tmp/1'

In [ ]:


cmd = ["git", "clone", github_url, clone_path]

try:
    subprocess.run(
        cmd,
        check=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        # timeout=timeout_sec,
    )
    # return final_path
    print(clone_path)
except FileNotFoundError as e:
    raise RuntimeError("git is not installed or not on PATH.") from e
except subprocess.  as e:
    # Clean partial clone
    shutil.rmtree(final_path, ignore_errors=True)
    raise RuntimeError("git clone timed out.") from e
except subprocess.CalledProcessError as e:
    # Clean partial clone
    # shutil.rmtree(final_path, ignore_errors=True)
    msg = (e.stderr or e.stdout or "").strip()
    raise RuntimeError(f"git clone failed: {msg if msg else 'unknown error'}") from e

/tmp/1


In [29]:
with tempfile.TemporaryDirectory() as tmp:
    tmp = '/tmp/2'
    cmd = ["git", "clone", github_url, tmp]
    try:
        subprocess.run(
            cmd,
            check=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            # timeout=timeout_sec,
        )
        print(glob(tmp))

    except FileNotFoundError as e:
        raise RuntimeError("git is not installed or not on PATH.") from e

CalledProcessError: Command '['git', 'clone', 'https://github.com/handong1587/handong1587.github.io', '/tmp/2']' returned non-zero exit status 128.

In [153]:
# features

In [154]:
repo_profile = {}

In [155]:
readme_file_paths = glob(os.path.join(tmp, "**/README.md"), recursive=True)
readme_files = {}
for path in readme_file_paths:
    with open(path) as f:
        path = path.split(tmp)[1]
        readme_files[path] = f.read()
repo_profile['readme_files'] = readme_files

In [157]:
tld_files = os.listdir(tmp)
repo_profile['top_level_directory_files'] = tld

In [158]:

class RepoSummary(BaseModel):
    summary: str = Field(description='A human-readable description of what the project does')
    technologies: list[str] = Field(description='List of main technologies, languages, and frameworks used')
    structure: str = Field(description='Brief description of the project structure')

schema = RepoSummary.model_json_schema()

system_prompt = f'''
You are given a Repo Profile; produce a concise architecture + tech summary,
Required information schema: {schema}
'''.strip()


completion = client.chat.completions.create(
    model=model,
    messages=[
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": json.dumps(repo_profile)
        }
    ],
    extra_body={
        "guided_json": schema,
    },

    # temperature=0.6,
    # max_tokens=51,
    # top_p=0.9
)

response = RepoSummary(**json.loads(completion.choices[0].message.content))


In [159]:
json.loads(response.model_dump_json())

{'summary': 'A personal blog showcasing papers, projects, websites, blogs read/studied, organized using a GitHub blog theme.',
 'technologies': ['GitHub', 'Markdown', 'Styling CSS'],
 'structure': 'Hierarchical directory structure with sub-directories for blog posts, projects, and research resources. README files provide details on the project and specific services provided'}

In [ ]:
# response = client.chat.completions.create(
#     model="google/gemma-2-2b-it",
#     messages=[
#         {
#             "role": "system",
#             "content": system_prompt
#         },
#         {
#             "role": "user",
#             "content": [
#                 {
#                     "type": "text",
#                     "text": json.dumps(repo_profile),
#                 }
#             ]
#         }
#     ]
# )

# print(response.to_json())

{
  "id": "chatcmpl-19454016058f45f58fc72b8c6ccd9f26",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "```json\n{\"summary\": \"This repository contains two different versions of the VGG network and a repository that facilitates training on VGG-16 and VGG-19 models.\", \"technologies\": [\"HTML\", \"JavaScript\", \"Sublime Text\"], \"structure\": \"This repository is a collection of README files describing different versions of the VGG model. It includes both old and newer implementations and proposes addressing training for the model via a linked GitHub repository.\"}\n```",
        "refusal": null,
        "role": "assistant",
        "annotations": null,
        "audio": null,
        "function_call": null,
        "tool_calls": [],
        "reasoning_content": null
      },
      "stop_reason": 107
    }
  ],
  "created": 1771875505,
  "model": "google/gemma-2-2b-it",
  "object": "chat.completion",

In [89]:
response.choices[0].message.content[8:]

'{"summary": "This repository contains two different versions of the VGG network and a repository that facilitates training on VGG-16 and VGG-19 models.", "technologies": ["HTML", "JavaScript", "Sublime Text"], "structure": "This repository is a collection of README files describing different versions of the VGG model. It includes both old and newer implementations and proposes addressing training for the model via a linked GitHub repository."}\n```'